# 02 — split v2: four slices, by video, stratified by procedure

Rung 01 (infra, not a scored rung). Rebuilds the frozen partition as **`train` / `val_id` /
`test_id` / `val_ood`**, superseding `frame_ood_v1.csv` for everything trained from
2026-08-19 on. v1 is NOT deleted — it is what every earlier result was measured under.

**Why v2 exists.** v1 held out 38 videos. Rung 42 promoted 30 of them into training, leaving
**8** — 6 lapchole (483 q) + 2 Sigmoid (800 q). Every rung since has been measured on those 8,
and on 2026-08-19 the epoch CI showed what that costs: running the rung-47 epoch effect through
`frame.metrics` (which clusters on video), **not one cell excluded zero** — `ALL_ID` +0.0290
[-0.0145, +0.0705], `object_recognition_ID` +0.0513 [-0.0022, +0.1140]. The point estimates
reproduced; the intervals did not. Eight clusters cannot resolve a 3-point effect.

**The design.**

| slice | videos | what it is for |
|---|---:|---|
| `train` | 72 | training |
| `val_id` | 24 | epoch / checkpoint selection |
| `test_id` | 24 | reported once, at the end |
| `val_ood` | 10 | the unseen PROCEDURE (Sigmoid), read-only |

- **Split by video count, not question mass.** heico videos carry 400 questions, lapchole ~80.
  CI width is set by CLUSTERS, so the count is what gets partitioned; question mass only breaks
  ties. It lands at 3209 / 3208 questions in `val_id` / `test_id` — one question apart.
- 🔴 **`val_ood` keeps its name for compatibility, not accuracy.** `metrics._distribution`,
  `delta.py` and `ledger.py` hardwire the literal string; renaming it would silently label
  every video "ID". It means **unseen procedure**, which is NOT the platform's OOD axis
  (centre). Our Sigmoid cell read 0.4086 locally against 0.6064 on the platform. Never report
  it as a proxy for the platform's OOD half — the manifest carries `ood_axis=procedure` to say
  so in the file itself.

**No challenge data is read.** v1 already lists every video with its procedure and question
count, so v2 regenerates on any machine, with no dataset and no GPU.

In [ ]:
# ── bootstrap ─────────────────────────────────────────
import sys, logging
from pathlib import Path

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", datefmt="%H:%M:%S")

import pandas as pd
from frame import split as sp

In [ ]:
# ── config (inline, this cell IS the run) ─────────────
SPLITS   = REPO / "experiments" / "splits"
V1       = SPLITS / "frame_ood_v1.csv"          # source of truth for the video inventory
V2       = SPLITS / "frame_split_v2.csv"        # what this notebook writes
SEED     = 42
OOD_PROC = "Sigmoid Resection"                  # left whole, before anything else is touched
FRACS    = {"train": 0.60, "val_id": 0.20, "test_id": 0.20}

out = sp.build_split_v2(V1, out_path=V2, seed=SEED, ood_procedure=OOD_PROC, fracs=FRACS)
print(out.groupby("split").agg(videos=("video_id", "count"), questions=("n_questions", "sum")).to_string())

In [ ]:
# ── what landed where ─────────────────────────────────
piv = out.pivot_table(index=["dataset", "procedure_type"], columns="split",
                      values="n_questions", aggfunc=["count", "sum"], fill_value=0)
print(piv.to_string())

In [ ]:
# ── guards: a manifest is only worth freezing if these hold ──
vs = sp.load_manifest(V2, verify=True)          # recomputes the sha256 against the sidecar
sp.assert_no_leak(vs)                           # train ∩ (everything else) = ∅

assert len(vs) == 130, len(vs)
assert out.n_questions.sum() == 20_000, out.n_questions.sum()

counts = pd.Series(list(vs.values())).value_counts().to_dict()
assert counts == {"train": 72, "val_id": 24, "test_id": 24, "val_ood": 10}, counts

ood = {k for k, s in vs.items() if s == "val_ood"}
sig = {(r.dataset, r.video_id) for _, r in out[out.procedure_type == OOD_PROC].iterrows()}
assert ood == sig, "val_ood must be EXACTLY the unseen procedure"

# every procedure that is NOT the held-out one must appear in all three id slices
for proc, g in out[out.procedure_type != OOD_PROC].groupby("procedure_type"):
    assert set(g.split) == {"train", "val_id", "test_id"}, (proc, sorted(set(g.split)))

print("all guards passed ·", counts)